# 03. 원격탐사 식생지수 — NDVI

**NDVI(정규식생지수)** 는 위성영상에서 식생의 분포·건강도를 파악하는 대표 지표입니다.
식물은 근적외선(NIR)을 강하게 반사하고 빨강(Red)을 흡수하는 성질을 이용합니다.

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

값은 -1 ~ 1이며, 높을수록 식생이 건강·밀집합니다.

1. NIR·Red 밴드 준비
2. NDVI 계산
3. 식생 분류 및 시각화

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

## 1. NIR·Red 밴드 준비

실제로는 위성영상에서 밴드를 읽습니다(`src.read(밴드번호)`).
여기서는 개념을 명확히 보기 위해 세 영역(식생·물·도시)을 가진 합성 밴드를 만듭니다.

In [ ]:
H = W = 200
red = np.full((H, W), 0.3, dtype='float32')
nir = np.full((H, W), 0.3, dtype='float32')

# 식생 영역: NIR 높고 Red 낮음 (NDVI 높음)
red[20:90, 20:90] = 0.1; nir[20:90, 20:90] = 0.7
# 물 영역: 둘 다 낮음 (NDVI 음수)
red[120:180, 30:90] = 0.05; nir[120:180, 30:90] = 0.03
# 도시 영역: 둘 다 중간 (NDVI 0 부근)
red[60:140, 120:180] = 0.35; nir[60:140, 120:180] = 0.40

red += 0.02*np.random.randn(H, W); nir += 0.02*np.random.randn(H, W)
print('밴드 준비 완료')

## 2. NDVI 계산

밴드 간 단순 사칙연산으로 NDVI를 계산합니다 (numpy 브로드캐스팅). 0으로 나누는 것을 방지합니다.

In [ ]:
ndvi = (nir - red) / (nir + red + 1e-9)
ndvi = np.clip(ndvi, -1, 1)
print(f'NDVI 범위: {ndvi.min():.2f} ~ {ndvi.max():.2f}')

plt.figure(figsize=(6, 5))
im = plt.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, label='NDVI'); plt.title('NDVI 지도'); plt.axis('off')
plt.show()

## 3. 식생 분류

NDVI 값을 구간으로 나눠 토지 유형을 분류합니다 (물/비식생 · 보통 · 건강한 식생).

In [ ]:
classified = np.zeros_like(ndvi, dtype=int)
classified[ndvi > 0.2] = 1    # 보통 식생
classified[ndvi > 0.5] = 2    # 건강한 식생

cmap = mcolors.ListedColormap(['#4a90d9', '#c2d96a', '#1a7a1a'])
labels = ['물/비식생', '보통 식생', '건강한 식생']

plt.figure(figsize=(6, 5))
im = plt.imshow(classified, cmap=cmap, vmin=0, vmax=2)
cbar = plt.colorbar(im, ticks=[0.33, 1, 1.67])
cbar.ax.set_yticklabels(labels)
plt.title('NDVI 식생 분류'); plt.axis('off')
plt.show()

for i, lab in enumerate(labels):
    print(f'{lab}: {(classified==i).sum()} 픽셀')

### 정리

- 근적외선·빨강 밴드로 NDVI를 계산해 식생 분포를 파악하고, 구간 분류로 토지 유형을 나눴습니다.
- NDVI는 농업 모니터링, 산불·가뭄 탐지, 토지피복 변화 분석 등에 폭넓게 쓰입니다.
- 실제로는 Landsat·Sentinel 위성영상의 밴드를 rasterio로 읽어 같은 방식으로 계산합니다.